# Chromosome painting: ChromoPainter and fineSTRUCTURE

**Purpose.** ADMIXTURE and PCA treat SNPs as independent. **Chromosome painting** uses the
haplotypes instead: each individual's genome is painted as a mosaic of chunks copied from
the others, and who copies from whom is far more sensitive to recent, fine-scale structure
than allele frequencies are. fineSTRUCTURE then clusters individuals from that painting.

**What you will do**
 - paint every individual against all the others with **ChromoPainter**
 - cluster the individuals from the resulting chunk counts with **fineSTRUCTURE** MCMC
 - build a tree of the inferred clusters and check that the MCMC has converged by
   comparing two independent runs
 - paint a target population against surrogates and infer its ancestry with
   **GLOBETROTTER** and **SOURCEFIND**

**The data.** 16 present-day populations, **256 individuals**, chromosome 22:

| Region | Populations |
|---|---|
| Africa | BantuKenya 11, BantuSouthAfrica 8, Mandenka 22, MbutiPygmy 13 |
| Central South Asia | Balochi 21, Burusho 25, Kalash 23, Makrani 22, Pathan 22 |
| East Asia | HanNchina 10, Mongola 10 |
| Europe | English 6, NorthItalian 12, Orcadian 15, Sardinian 28, Tuscan 8 |

plus a **simulated admixed group of 20 individuals**: 80% Brahui (Pakistan) and 20% Yoruba
(Nigeria), admixing 30 generations ago. Because it is simulated, you know what the methods
should recover.

The data are **phased haplotypes** with a recombination map — painting needs both, which
is what separates it from the frequency-based methods.

**Note.** This exercise **compiles the software from source**, and the fineSTRUCTURE MCMC
takes several minutes. The full ChromoPainter run is left commented out for the same
reason, with its output provided.

**Leads to** [Dating admixture](dating_admixture_human.ipynb), which uses the ChromoPainter
output from part 2.

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data moves, this is the ONLY cell you need to change.
# No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA=/course/data/current_data/chromopainter

# where you will do the exercise
WORK_DIR=$HOME/chromopainter_finestructure

mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.chromopainter_workdir
cd $WORK_DIR

echo --- unpacking the practical files ---
tar -xzf $DATA/FineStructurePractical.tar.gz -C .

echo; echo --- folders ---
ls

## 1 Clustering individuals: CHROMOPAINTER and fineSTRUCTURE

First we will apply **ChromoPainter** and **fineSTRUCTURE** to cluster individuals. For simplicity, we will only cluster based on chromosome 22 data.

Unarchive the file for practice `FineStructurePractical.tar.gz`. Then navigate to the folder `FineStructureFiles/`.

In [ ]:
# the practical files were unpacked in the setup cell
ls


Extract `ChromoPainterv2` and `fineSTRUCTURE`:

In [ ]:
tar -xzvf ChromoPainterv2.tar.gz
unzip fs_4.0.0.zip

We will use the pre-compiled binary `fs_linux_glibc2.3` in the directory `fs_4.0.0/`.

Compile ChromoPainterv2 with:

In [ ]:
gcc -o ChromoPainterv2 ChromoPainterv2.c -lm -lz

**Question**
 - ChromoPainter is compiled here from a single C file. What does it need besides the genotypes that ADMIXTURE and PCA do not?

We aim to cluster all individuals in the above table. To do so, we first use `ChromoPainter`
to paint each individual from these populations against the others:

In [ ]:
### Please delete all the leading "#" sign at each of following #lines if you really want to run 
### the command (it takes 10 min)

#./ChromoPainterv2 -g example/BrahuiYorubaSimulationChrom22.haplotypes \
#    -r example/BrahuiYorubaSimulationChrom22.recomrates \
#    -t example/BrahuiYorubaSimulation.idfile.txt \
#    -f BrahuiYorubaSimulationSurrogatesOnly.poplist.txt 0 0 \
#    -o example/BrahuiYorubaSimulationSurrogatesPaintingChrom22 \
#    -a 0 0 -s 0

(As mentioned in the lecture, note that you could initially do E-M steps to infer the
“switch” (-n) and “mutation” (-M) parameters, but we will instead use default values. In
most applications, skipping this E-M step will not make much or any difference, but it is
good practice!).

**A problem** – it may be too slow for this practical, as it takes **≈10min**. Therefore I have
already done this painting for you in `data/BrahuiYorubaSimulationSurrogatesPaintingChrom22*`

In [ ]:
ls data/BrahuiYorubaSimulationSurrogatesPaintingChrom22*

**Questions**
 - The painting produces `.chunkcounts.out` — a matrix of how much each individual copies from each other individual. Why is that matrix more informative than allele frequencies?
 - Is the matrix symmetric? Should it be?

Next we will run fineSTRUCTURE to cluster individuals based on the
`data/BrahuiYorubaSimulationSurrogatesPaintingChrom22.chunkcounts.txt` output
file. This file gives the total number of haplotype segments (“chunks”) that each recipi-
ent individual copies from each donor individual. To do so, we first need to calculate a
nuisance parameter “c”, using:

In [ ]:
# calcC_Continents.R prints the 'c' value that finestructure needs.
# capture it in a variable rather than copying the number by hand.
C=$(Rscript calcC_Continents.R data/BrahuiYorubaSimulationSurrogatesPaintingChrom22 | tail -n 1 | tr -d '[:space:]')
echo "c = $C"
echo $C > c_value.txt


The value printed above is the `c` parameter that `finestructure` needs. It was captured
into the shell variable `$C`, so the commands below use the number you just computed
rather than one typed in by hand.

**Question**
 - What is `c` correcting for? (hint: chunks copied from a close relative are not
   independent observations)

In [ ]:
fs_4.0.0/fs_linux_glibc2.3 finestructure -I 1 -c $C \
    -x 10000 -y 20000 -z 100 \
    data/BrahuiYorubaSimulationSurrogatesPaintingChrom22.chunkcounts.out \
    BrahuiYorubaSimulationSurrogatesPaintingChrom22.finestructure.out

**Questions**
 - `-x 10000 -y 20000` sets burn-in and sampling iterations. What is the MCMC exploring?
 - Why does fineSTRUCTURE need MCMC at all, when ADMIXTURE just optimises?

(Note that in real applications, you should probably have each of ‘‘-x’’, ‘‘-y’’,
‘‘-z’’ a factor of 100 higher.)

To generate a tree using this output, type:

In [ ]:
fs_4.0.0/fs_linux_glibc2.3 finestructure -c $C -x 10000 -k 2 -m T -t 1000000 \
    data/BrahuiYorubaSimulationSurrogatesPaintingChrom22.chunkcounts.out \
    BrahuiYorubaSimulationSurrogatesPaintingChrom22.finestructure.out \
    BrahuiYorubaSimulationSurrogatesPaintingChrom22.finestructureTREE.out

**Question**
 - This builds a tree of the inferred clusters. How is that different from a population tree such as the one from neighbour joining?

(Note that in real applications you should probably have ‘‘`-x`’’ a factor of 10 higher.)

We will also make a “coincidence matrix” that gives the proportion of MCMC samples
for which each pair of individuals is clustered together:

In [ ]:
fs_4.0.0/fs_linux_glibc2.3 finestructure -c $C -e meancoincidence \
    data/BrahuiYorubaSimulationSurrogatesPaintingChrom22.chunkcounts.out \
    BrahuiYorubaSimulationSurrogatesPaintingChrom22.finestructure.out \
    BrahuiYorubaSimulationSurrogatesPaintingChrom22.finestructureCOINCIDENCE.out

A good way to assess whether you have done enough MCMC samples is to run **fineSTRUCTURE** again, using a different seed (e.g. with “`-s 2`”):

In [ ]:
fs_4.0.0/fs_linux_glibc2.3 finestructure -s 2 -I 1 -c $C \
    -x 10000 -y 20000 -z 100 \
    data/BrahuiYorubaSimulationSurrogatesPaintingChrom22.chunkcounts.out \
    BrahuiYorubaSimulationSurrogatesPaintingChrom22.finestructureSEED2.out \

fs_4.0.0/fs_linux_glibc2.3 finestructure -c $C -x 10000 -k 2 -m T -t 1000000 \
    data/BrahuiYorubaSimulationSurrogatesPaintingChrom22.chunkcounts.out \
    BrahuiYorubaSimulationSurrogatesPaintingChrom22.finestructureSEED2.out \
    BrahuiYorubaSimulationSurrogatesPaintingChrom22.finestructureSEED2TREE.out \

fs_4.0.0/fs_linux_glibc2.3 finestructure -c $C -e meancoincidence \
    data/BrahuiYorubaSimulationSurrogatesPaintingChrom22.chunkcounts.out \
    BrahuiYorubaSimulationSurrogatesPaintingChrom22.finestructureSEED2.out \
    BrahuiYorubaSimulationSurrogatesPaintingChrom22.finestructureSEED2COINCIDENCE.out

**Questions**
 - This is a second MCMC run from a different seed. Why run it at all?
 - If the two runs disagree, what would you change?

Finally we will plot some results using R scripts I have provided. Use `CHROMOPAINTERHeatMapPlot.R`
to plot a heatmap of the `CHROMOPAINTER_chunkcounts.out` output, with individuals
clustered according to the results of the initial **fineSTRUCTURE** run:

In [ ]:
R CMD BATCH CHROMOPAINTERHeatMapPlot.R

**Question**
 - In the heat map, what do the rows and columns represent, and what does a bright block on the diagonal mean?

This will make a new file called
`BrahuiYorubaSimulationSurrogatesPaintingChrom22HEATMAPWithTree.pdf`, which contains a heatmap giving the total number of “chunks” (haplotype segments) that each recipient individual (column) copies from each donor individual (row). The tick marks along each axis color individuals based on their population labels (see legend at bottom)

Use `FineStructureCoincidenceMatrixVisualize2Seeds.R` to plot the coincidence ma-
trix for both fineSTRUCTURE runs:

In [ ]:
R CMD BATCH FineStructureCoincidenceMatrixVisualize2Seeds.R

**Questions**
 - The coincidence matrix shows how often two individuals were put in the same cluster across the two runs. What does a blurry block tell you?
 - Which groups are cleanly separated and which are not?

This will make a new file called
`BrahuiYorubaSimulationSurrogatesPaintingChrom22FSCoincidencePlot.pdf`, which
contains a heatmap giving the proportion of MCMC samples that each individual (rows)
is clustered with every other individual (columns). The top left and bottom right triangles give these proportions for the first and second finestructure runs, respectively.
Individuals are ordered along the axes according to the inferred finestructure tree from
the first run, i.e. ordered as in the CHROMOPAINTER heatmap.

Use these plots to answer the following questions:

1. Which groups are copied (painted from) least by the other groups?
2. Which groups copy the most from each other?
3. Do the inferred clusters seem sensible?
4. Does the inferred tree seem sensible?
5. How consistent do results from the two runs appear to be?

## 2 Inferring ancestry: GLOBETROTTER and SOURCEFIND

Next we will use **GLOBETROTTER** and **SOURCEFIND** to infer ancestry proportions for the
simulated population. This will make use of the painting of the ancestry surrogate populations that we did in the previous section.

We first need to paint the simulated target individuals against these surrogate populations:

In [ ]:
./ChromoPainterv2 -g example/BrahuiYorubaSimulationChrom22.haplotypes \
    -r example/BrahuiYorubaSimulationChrom22.recomrates \
    -t example/BrahuiYorubaSimulation.idfile.txt \
    -f BrahuiYorubaSimulation.poplistReduced.txt 0 0 \
    -o example/BrahuiYorubaSimulationAdmixtureChrom22 -s 10

**Question**
 - This paints the admixed target against the surrogate populations rather than everyone against everyone. Why is that the right painting for inferring ancestry?

The output file of interest here is
`example/BrahuiYorubaSimulationAdmixtureChrom22.chunklengths.out`, which gives
the total (cM) amount of DNA across chromosome 22 that a target individual copies from
each donor poplation. We will combine this painting with that of the surrogates, using a
script I made:

In [ ]:
R CMD BATCH CHROMOPAINTERSurrogateTargetPaintingsCombine.R

Next unarchive `GLOBETROTTER.tar.gz`:

In [ ]:
tar -xzvf GLOBETROTTER.tar.gz

and compile with:

In [ ]:
R CMD SHLIB -o GLOBETROTTERCompanion.so GLOBETROTTERCompanion.c -lz

Run `GLOBETROTTER` using `BrahuiYorubaSimulationAdmixture.paramfileNNLS.txt`, which
specifies (using `num.mixing.iterations:0`) that we only want to run the NNLS model
in `GLOBETROTTER` to infer ancestry proportions in the simulated population, and not infer
or date admixture:

In [ ]:
R < GLOBETROTTER.R BrahuiYorubaSimulationAdmixture.paramfileNNLS.txt --no-save > output.out

**Questions**
 - GLOBETROTTER reports sources, proportions and a date. How close are they to the true 80% Brahui / 20% Yoruba, 30 generations?
 - Which surrogate did it pick for each source, and is that the population you would have chosen?

This will make the output file `example/BrahuiYorubaSimulationAdmixed.GTnnls.main.txt`,
which contains the inferred ancestry proportions under the NNLS model.

Now run `SOURCEFIND` using `BrahuiYorubaSimulationAdmixture.SourcefindParamfile.txt`:

In [ ]:
tar -xzvf SOURCEFINDv2.tar.gz

R < sourcefindv2.R BrahuiYorubaSimulationAdmixture.SourcefindParamfile.txt --no-save >output.out

**Questions**
 - SOURCEFIND estimates the mixture proportions differently. Does it agree with GLOBETROTTER?
 - If two methods disagree on the same data, how would you decide which to believe?

This will make the output file `BrahuiYorubaSimulation.sourcefind.txt`, which contains the inferred ancestry proportions under `SOURCEFIND`.

Answer the following questions.

1. How well does the GLOBETROTTER NNLS soluation capture the ancestry of the simulated population?

2. Find the SOURCEFIND MCMC sample with the highest posterior probability. What does this show? How does its inference compare to that of the other MCMC samples (or the mean across samples)?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/admixture/quiz/chromopainter.json")
